# Inpactor3-SegNT · Entrenamiento en Colab

Genome Language Model (Nucleotide Transformer 50M) + cabeza de segmentación
para detectar LTR-RTs en Arabidopsis Chr1.

**Datos**: Chr1 dividido en 3 pseudo-scaffolds con 10/3/3 anotaciones
de Inpactor2 (piloto). Cuando corramos Inpactor2 sobre TAIR10 completo
podremos hacer split por cromosoma real.

**Antes de correr**: `Runtime → Change runtime type → T4 GPU`.

Tiempo total estimado: **~20-30 min** en GPU T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar dependencias

`pip install -e .` instala el paquete `inpactor3_segnt` en modo editable,
así `python -m inpactor3_segnt.train` funciona sin PYTHONPATH.

In [ ]:
%cd /content
!git clone https://github.com/Inpactor3/Inpactor3-SegNT.git
%cd Inpactor3-SegNT
!pip install -q -e .

## 3. Verificar datos y anotaciones ground truth

In [ ]:
!ls -lh data/raw/
!echo '---'
!head -3 data/raw/Inpactor2_predictions_pseudo.tab
!echo '---'
!wc -l data/raw/Inpactor2_predictions_pseudo.tab

## 4. Entrenar SegNT (10 épocas, ~15-25 min GPU T4)

In [ ]:
!python -m inpactor3_segnt.train --config configs/nt_50m_chr1.yaml

## 5. Inferencia sobre pseudo-scaffold de test (Chr1_test = últimos 3 Mb de Chr1)

In [ ]:
!python -m inpactor3_segnt.predict \
    --checkpoint models/best.pt \
    --genome data/raw/TAIR10_pseudo.fasta \
    --scaffold Chr1_test \
    --out results/chr1_test_predictions.tab

## 6. Evaluar contra Inpactor2 (métrica del informe: F1 por ventana)

In [ ]:
!python -m inpactor3_segnt.evaluate \
    --pred results/chr1_test_predictions.tab \
    --truth data/raw/Inpactor2_predictions_pseudo.tab \
    --scaffold Chr1_test \
    --genome data/raw/TAIR10_pseudo.fasta \
    --report results/chr1_test_report.md

In [ ]:
!cat results/chr1_test_report.md

## 7. Descargar checkpoint y reportes a tu PC

In [ ]:
from google.colab import files
files.download('models/best.pt')
files.download('results/chr1_test_report.md')
files.download('results/chr1_test_predictions.tab')